# ⚫Automated Military Data Scraping and Dataset Creation





#Objective:
The purpose of this phase is to streamline and automate the data gathering process. Rather than manually collecting military statistics for more than 140 countries, a Python-based approach is implemented to systematically access the Global Firepower index. The script retrieves country-specific URLs, extracts essential indicators such as manpower, defense expenditure, and equipment totals, and consolidates the information into a structured raw dataset ready for further processing.

# ⚙️ Step 1 — Read URLs from file

This section handles the initial connection to the Global Firepower and getting country's list.

In [ ]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup

In [ ]:
#1. Get Country List
web = requests.get("https://www.globalfirepower.com/country-military-strength-detail.php?country_id=india").text
soup = BeautifulSoup(web, 'lxml')
soup.prettify()

financial = soup.find_all('div', class_='specsGenContainers')
d_f = {}

for f in financial:
    # Fun Fact: if u take 'textLarge.textYellow.textBold.textShadow'
    # You will miss
    tag = f.select_one('span.textLarge.textBold.textShadow')
    value_tag = f.select('span.textWhite.textShadow')

    if tag and value_tag:
        name = tag.text.strip()
        value = value_tag[-1].text.strip()
        d_f[name] = value
# Verify It
d_f


{'Purchasing Power Parity:': '$14,244,000,000,000 USD',
 'Foreign Exchange/Gold:': '$643,043,000,000 USD',
 'Defense Budget:': '$109,000,000,000 USD',
 'External Debt:': '$212,728,000,000 USD',
 'Square Land Area:': '3,287,263 km',
 'Coastline Coverage:': '7,000 km',
 'Shared Borders:': '13,888 km',
 'Waterways (usable):': '14,500 km',
 'Total Population:': '1,409,128,296',
 'Available Manpower': '662,290,299',
 'Fit-for-Service': '522,786,598',
 'Reaching Mil Age Annually': '23,955,181',
 'Active Personnel': '1,431,000',
 'Reserve Personnel': '1,000,000',
 'Paramilitary': '2,527,000',
 'Air Force Personnel*': '289,000',
 'Army Personnel*': '2,148,000',
 'Navy Personnel*': '148,869',
 'Yearly Mobilization Potential': '23,652,761',
 'Aircraft Total:': 'Stock: 2,183\t\t\t\nReadiness: 1,637*',
 'Fighters:': 'Stock: 476 (21.8%)\n\nReadiness: 357*',
 'Attack Types:': 'Stock: 124 (5.7%)\n\nReadiness: 93*',
 'Transports (Fixed-Wing):': 'Stock: 277 (12.7%)\n\nReadiness: 208*',
 'Trainers:': 'S

In [ ]:
#2. Send requests to website
webpage = requests.get("https://www.globalfirepower.com/countries-listing.php").text
# Create object of BeautifulSoup, will be used for parsing
soup = BeautifulSoup(webpage, 'lxml')

# 🧬 Step 2 — Loop through URLs and Collect data

This sections iterates to scrape each country's IDs ,and save the final output as CSV.

---




In [ ]:
#1. Reading Country's IDs
country = soup.find_all('a', href = True)
ids = []
for a in country:
    href = a["href"]
    if "country-military-strength-detail.php?country_id=" in href:
        cid = href.split("country_id=")[1]
        ids.append(cid)
ids = list(set(ids))

In [ ]:
#2. Mapping
key_map = {
    "gdp": "Purchasing Power Parity:",
    "total_population": "Total Population",
    "defense_budget": "Defense Budget:",
    "total_manpower": "Available Manpower",
    "active_personnel": "Active Personnel",
    "reserve_personnel": "Reserve Personnel",
    "air_force_personnel": 'Air Force Personnel*',
    "army_personnel": 'Army Personnel*',
    "navy_personnel": 'Navy Personnel*',

    "total_aircraft": "Aircraft Total:",
    "attack_aircraft": "Attack Types:",
    "fighter_aircraft": "Fighters:",
    "transport_aircraft": "Transports (Fixed-Wing):",
    "helicopters": "Helicopters:",
    "special_mission_aircraft": "Special-Mission:",
    "trainer_aircraft": "Trainers:",
    "attack_helicopters": "Attack Helicopters:",
    "tanker_aircraft": "Tanker Fleet:",

    "naval_assets": "Total Assets:",
    "aircraft_carriers": 'Aircraft Carriers:',
    "helicopter_carriers": 'Helicopter Carriers:',
    "destroyers": 'Destroyers:',
    "frigates": 'Frigates:',
    "corvettes": 'Corvettes:',
    "submarines": 'Submarines:',
    "patrol_vessel": 'Patrol Vessels:',
    "mine_warfare": 'Mine Warfare:',

    "tanks": "Tanks:",
    "self_propelled_artillery": "Self-Propelled Artillery:",
    "towed_artillery": "Towed Artillery:",
    "rocket_artillery": "MLRS (Rocket Artillery):",
    "defense_budget": "Defense Budget:",
    "external_debt": "External Debt:"
}

In [ ]:
#3. Automation Function
def automation(ids, key_map):
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_countries = {}

    for cid in ids:
        url = base_url + cid

        headers = {"User-Agent": "Mozilla/5.0"}
        web = requests.get(url, headers=headers).text
        soup = BeautifulSoup(web, "lxml")

        country_name = cid.replace("-", " ").title()

        power_index = None
        info_block = soup.select_one("span.textNormal.textDkGray")

        if info_block:
            match = re.search(r'\d+\.\d+', info_block.text)
            if match:
                power_index = match.group()

        financial = soup.find_all('div', class_='specsGenContainers')
        d_f = {}

        for f in financial:
            tag = f.select('span.textLarge.textBold.textShadow')
            value_tags = f.select('span.textWhite.textShadow')

            if tag and value_tags:
                name = tag[0].text.strip()
                values = [v.text.strip() for v in value_tags]
                d_f[name] = values

        result = {}

        for new_key, old_key in key_map.items():
            if old_key in d_f:
                result[new_key] = d_f[old_key][-1]

        result["power_index"] = power_index
        all_countries[country_name] = result

    return all_countries

In [ ]:
#4. Execution
all_countries = automation(ids, key_map)
all_countries

df = pd.DataFrame.from_dict(all_countries, orient="index")
df.reset_index(inplace=True)
df.rename(columns={"index":"country"}, inplace=True)
df.head()

,country,gdp,defense_budget,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,total_aircraft,...,corvettes,submarines,patrol_vessel,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,external_debt,power_index
0,Singapore,"$800,304,000,000 USD","$18,000,000,000 USD","3,918,498","51,000","252,500","13,500","280,000","9,000",Stock: 235\t\t\t\nReadiness: 176*,...,6,6,55,4,Stock: 170\t\t\t\t\nReadiness: 128*,Stock: 48\t\t\t\t\nReadiness: 36*,Stock: 89\t\t\t\t\nReadiness: 67*,Stock: 24\t\t\t\t\nReadiness: 18*,"$1,884,751,660,000 USD",0.5272
1,Uzbekistan,"$379,989,000,000 USD","$6,210,000,000 USD","18,990,708","120,000",0,"15,000","95,000",550,Stock: 159\t\t\t\nReadiness: 87*,...,0,0,0,0,Stock: 470\t\t\t\t\nReadiness: 259*,Stock: 177\t\t\t\t\nReadiness: 97*,Stock: 236\t\t\t\t\nReadiness: 130*,Stock: 108\t\t\t\t\nReadiness: 59*,"$25,714,000,000 USD",0.9908
2,Argentina,"$1,213,000,000,000 USD","$993,919,790 USD","20,677,529","108,000","12,370","14,000","108,000","18,370",Stock: 241\t\t\t\nReadiness: 133*,...,6,2,13,0,Stock: 362\t\t\t\t\nReadiness: 199*,Stock: 20\t\t\t\t\nReadiness: 11*,Stock: 172\t\t\t\t\nReadiness: 95*,Stock: 27\t\t\t\t\nReadiness: 15*,"$74,362,000,000 USD",0.5983
3,El Salvador,"$73,961,000,000 USD","$1,037,200,000 USD","3,181,777","25,000",0,"2,000","20,500","2,000",Stock: 51\t\t\t\nReadiness: 28*,...,0,0,8,0,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 76\t\t\t\t\nReadiness: 49*,Stock: 0\t\t\t\t\nReadiness: 0*,"$12,668,000,000 USD",2.9973
4,Tanzania,"$246,706,000,000 USD","$1,476,113,361 USD","12,143,182","25,000",0,"1,825","21,975","1,200",Stock: 38\t\t\t\nReadiness: 17*,...,0,0,10,0,Stock: 200\t\t\t\t\nReadiness: 100*,Stock: 0\t\t\t\t\nReadiness: 0*,Stock: 205\t\t\t\t\nReadiness: 103*,Stock: 70\t\t\t\t\nReadiness: 35*,"$17,513,000,000 USD",1.9111


In [ ]:
#5. Exporting raw dataset
df.to_csv("military_raw_data.csv", index=False)
from google.colab import files
files.download("military_raw_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Takeaway:
Successfully extracted raw metrics for 145 countries and exported the unstructured dataset to military_raw_data.csv


# ⚫ Data Cleaning and standardization

#Objective:
The objective of this phase is to convert noisy scraped data into a clean,numeric,analytics-ready dataset

# Step 1- Load Raw data

Load input file and perform basic configuration

In [ ]:
import pandas as pd
import numpy as np
# Load raw CSV file
raw_df = pd.read_csv("military_raw_data.csv")
raw_df.to_excel("military_raw_data_copy.xlsx", index=False)

# Step 2- Standardize column names

Convert column names to lower_case and snake_case to ensure consistent naming across the dataset.

In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

#Step3- Identify Data Noise and Clean Numeric Columns

Check for and remove commas,%,+,and extra text.Convert the cleaned values to numeric types.

In [ ]:
# Numeric columns
numeric_cols = [
    "power_index_rank",
    "total_manpower",
    "active_personnel",
    "naval_assets",
    "tanks",
    "defense_budget",
    "gdp"
]

numeric_cols = [c for c in numeric_cols if c in df.columns]


# Clean numeric columns (VECTOR SAFE)
for col in numeric_cols:
    series = df[col].astype(str).str.lower()

    # detect billion / million
    billion_mask = series.str.contains("billion", na=False)
    million_mask = series.str.contains("million", na=False)

    # remove all non-numeric characters except dot
    series = series.str.replace(r"[^0-9.]", "", regex=True)

    series = pd.to_numeric(series, errors="coerce")

    # apply multipliers
    series.loc[billion_mask] = series.loc[billion_mask] * 1_000_000_000
    series.loc[million_mask] = series.loc[million_mask] * 1_000_000

    df[col] = series




#Step 4-Handling Missing Value

Identify columns with missing data.

In [ ]:
# Handle missing values
for col in numeric_cols:
    if df[col].isna().mean() < 0.02:
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(df[col].median())

# Drop rows without country
df = df.dropna(subset=["country"])


# Validation
print("Final shape:", df.shape)
print(df[numeric_cols].isna().sum())
print(df.info())


# Clean external_debt column
if "external_debt" in df.columns:
    df["external_debt"] = (
        df["external_debt"]
        .astype(str)
        .str.replace(r"[^0-9]", "", regex=True)
    )

    df["external_debt"] = pd.to_numeric(df["external_debt"], errors="coerce")

    # Handle missing values
    if df["external_debt"].isna().mean() < 0.02:
        df["external_debt"] = df["external_debt"].fillna(0)
    else:
        df["external_debt"] = df["external_debt"].fillna(df["external_debt"].median())


# Validation
print(df["external_debt"].head())
print(df["external_debt"].dtype)

Final shape: (145, 33)
total_manpower      0
active_personnel    0
naval_assets        0
tanks               0
defense_budget      0
gdp                 0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   country                   145 non-null    object 
 1   gdp                       145 non-null    int64  
 2   defense_budget            145 non-null    int64  
 3   total_manpower            145 non-null    int64  
 4   active_personnel          145 non-null    int64  
 5   reserve_personnel         145 non-null    object 
 6   air_force_personnel       145 non-null    object 
 7   army_personnel            145 non-null    object 
 8   navy_personnel            145 non-null    object 
 9   total_aircraft            145 non-null    int64  
 10  attack_aircraft           145 non-null    int64  
 11  fighter

#Step 5- Validate Cleaned Data

Check data types, null percentages, and ensure row count meets the expected threshold(140+ countries).

In [ ]:
import pandas as pd
import numpy as np
import re

# Columns with Stock / Readiness text
stock_columns = [
    "total_aircraft",
    "attack_aircraft",
    "fighter_aircraft",
    "transport_aircraft",
    "helicopters",
    "special_mission_aircraft",
    "trainer_aircraft",
    "attack_helicopters",
    "tanker_aircraft",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_artillery"
]

stock_columns = [c for c in stock_columns if c in df.columns]


# Function to extract STOCK number
def extract_stock(value):
    if pd.isna(value):
        return 0

    value = str(value)

    match = re.search(r"stock[:\s]*([0-9,]+)", value.lower())
    if match:
        return int(match.group(1).replace(",", ""))

    # fallback: first number
    fallback = re.search(r"\d+", value)
    if fallback:
        return int(fallback.group())

    return 0

# Apply extraction
for col in stock_columns:
    df[col] = df[col].apply(extract_stock)


# Validation
print(df[stock_columns].head())
print(df[stock_columns].info())

   total_aircraft  attack_aircraft  fighter_aircraft  transport_aircraft  \
0             235                0               100                   9   
1             159               13                26                  12   
2             241               11                23                  25   
3              51               15                 0                   3   
4              38                0                11                   6   

   helicopters  special_mission_aircraft  trainer_aircraft  \
0           75                         9                36   
1          101                         0                 7   
2           96                        13                71   
3           28                         0                10   
4           15                         0                 6   

   attack_helicopters  tanker_aircraft  self_propelled_artillery  \
0                  20               11                        48   
1                  34             

#Step 6- Export cleaned dataset

Save the fully numeric,standardized dataset as military_cleaned.csv.

In [ ]:
import pandas as pd

# Save final cleaned dataset
df.to_csv("military_cleaned.csv", index=False)

#Takeaway:
Cleaned 30 numeric columns,specially identifying and zero-imputing exactly 32 missing values across all naval asset fields.